# HNSW + 在线 HMM 检索与 Recall@25 评估

使用真实 NetVLAD 数据：多轨迹 Query，每条轨迹按帧做 HNSW Top-K 检索后经 OnlineHMM 时序平滑；
GT 采用坐标法（uav_infos.csv 经纬度 + 方圆 25 张），评估 Recall@25，对比「仅 HNSW」与「HNSW+HMM」。

In [1]:
import sys
import importlib
from pathlib import Path

_root = Path().resolve()
if _root.name == "hnsw_performance_analysis":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import faiss

from demo_readH5 import (
    load_netvlad_descriptors,
    load_multi_netvlad_descriptors,
    DB_H5_PATHS,
    QUERY_H5_PATHS,
)
from scene_config import (
    get_db_scene_ranges,
    get_query_trajectory_ranges,
    DATASETS_BASE,
)
import coords_gt_utils
importlib.reload(coords_gt_utils)
from coords_gt_utils import (
    build_database_coords,
    build_gt_9_for_all_trajectories,
)
from HMM.HMM import OnlineHMM

## 1. 加载 DB 与 Query（多轨迹）

In [2]:
print("加载 DB...")
db_names, db_descs = load_multi_netvlad_descriptors(DB_H5_PATHS)
db_descs = db_descs.astype(np.float32)
N_db, D = db_descs.shape
print(f"DB: {N_db} 条, 维度 {D}")

print("按轨迹加载 Query...")
query_per_traj = []
for h5_path in QUERY_H5_PATHS:
    names, descs = load_netvlad_descriptors(Path(h5_path))
    query_per_traj.append((names, descs.astype(np.float32)))

query_names = []
query_descs_list = []
for _, (names, descs) in enumerate(query_per_traj):
    query_names.extend(names)
    query_descs_list.append(descs)
query_descs = np.vstack(query_descs_list)
trajectory_ranges = get_query_trajectory_ranges(QUERY_H5_PATHS)
n_queries = query_descs.shape[0]
print(f"Query: {n_queries} 条, 共 {len(trajectory_ranges)} 条轨迹")
db_scene_ranges = get_db_scene_ranges(DB_H5_PATHS, db_names)

加载 DB...
DB: 11009 条, 维度 4096
按轨迹加载 Query...
Query: 5988 条, 共 15 条轨迹


## 2. DB 坐标（供 HMM 转移约束）与坐标法 GT@25

In [3]:
print("构建 database_coords（GDAL + id_startx_starty）...")
database_coords = build_database_coords(db_names, db_scene_ranges, DATASETS_BASE)
n_valid = np.sum(~np.any(np.isnan(database_coords), axis=1))
print(f"有效坐标: {n_valid}/{N_db}")

print("构建坐标法 GT@25（每 query 方圆 25 张）...")
gt_9_list = build_gt_9_for_all_trajectories(
    trajectory_ranges,
    db_scene_ranges,
    db_names,
    DATASETS_BASE,
)
n_with_gt = sum(1 for s in gt_9_list if len(s) > 0)
print(f"有 GT 的 query 数: {n_with_gt}/{len(gt_9_list)}")
if n_with_gt > 0:
    sizes = [len(s) for s in gt_9_list if len(s) > 0]
    print(f"GT 集合大小: min={min(sizes)}, max={max(sizes)}, mean={sum(sizes)/len(sizes):.1f}  (若 max>9 说明已按 25 张生成)")
    i0 = next(i for i in range(len(gt_9_list)) if len(gt_9_list[i]) > 0)
    gt0 = sorted(gt_9_list[i0])
    print(f"\n第一张有 GT 的 query（index={i0}）的 25 张 GT 图片:")
    for j, idx in enumerate(gt0):
        name = db_names[idx] if idx < len(db_names) else f"<{idx}>"
        print(f"  [{j+1:2d}] {name}")

构建 database_coords（GDAL + id_startx_starty）...
=== database_coords 各场景（未算出坐标的会打印原因）===
  [city1] 共 342 条 -> 有效坐标 342/342
  [city2] 共 168 条 -> 有效坐标 168/168
  [city3] 共 224 条 -> 有效坐标 224/224
  [industry1] 共 868 条 -> 有效坐标 868/868
  [industry2] 共 414 条 -> 有效坐标 414/414
  [industry3] 共 1476 条 -> 有效坐标 1476/1476
  [park1] 共 324 条 -> 有效坐标 324/324
  [rural1] 共 1845 条 -> 有效坐标 1845/1845
  [rural2] 共 1845 条 -> 有效坐标 1845/1845
  [rural3] 共 399 条 -> 有效坐标 399/399
  [school] 共 1178 条 -> 有效坐标 1178/1178
  [suburbs1] 共 480 条 -> 有效坐标 480/480
  [suburbs2] 共 468 条 -> 有效坐标 468/468
  [village1] 共 285 条 -> 有效坐标 285/285
  [village2] 共 693 条 -> 有效坐标 693/693
  -> 合计有效坐标: 11009/11009

有效坐标: 11009/11009
构建坐标法 GT@25（每 query 方圆 25 张）...
=== 坐标法 GT@25 各轨迹（未算出 GT 的会打印原因）===

--- 坐标法 GT 首帧诊断 [city1] ---
大图 mapbox geotransform (6 参数):
  gt[0] 左上角 X (投影/经度): 12123218.434142068
  gt[1] 像元宽:               0.2985821417389691
  gt[2] 旋转(常为 0):         0.0
  gt[3] 左上角 Y (投影/纬度): 4062398.742272254
  gt[4] 旋转(常为 0):         0.0
  

## 3. HNSW 索引（NetVLAD 已归一化，用内积）

In [4]:
K = 10
M = 24
ef_construction = 200
ef_search = 50

# 内积索引（归一化向量上等价于余弦）；Faiss 内积返回越大越相似
index = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search
index.add(db_descs)
print(f"HNSW 索引已构建: M={M}, efConstruction={ef_construction}, efSearch={ef_search}, K={K}")

HNSW 索引已构建: M=24, efConstruction=200, efSearch=50, K=10


## 4. 逐轨迹运行 HNSW + HMM，并计算 Recall@25

In [5]:
# 同场景检索：每条 query 只在该 query 所属场景的 DB 内检索（否则全局检索常命中其它场景，Recall@25 恒为 0）
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

index_per_scene = {}
for scene_name, db_start, db_end in db_scene_ranges:
    seg = db_descs[db_start:db_end]
    idx = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = ef_construction
    idx.hnsw.efSearch = ef_search
    idx.add(seg)
    index_per_scene[scene_name] = (idx, db_start, db_end)

pred_hnsw = np.full(n_queries, -1, dtype=np.int64)
pred_hmm = np.full(n_queries, -1, dtype=np.int64)
query_idx = 0

for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    idx_scene, db_start, db_end = index_per_scene.get(scene_name, (None, 0, 0))
    if idx_scene is None:
        query_idx += (end - start)
        continue
    q_descs = query_descs[start:end]
    n_frames = q_descs.shape[0]
    k_scene = min(K, db_end - db_start)
    D_t, I_local = idx_scene.search(q_descs, k_scene)
    I_t = I_local.astype(np.int64) + db_start
    dist_for_hmm = 1.0 - D_t.astype(np.float64)

    hmm = OnlineHMM(coords_for_hmm, K)
    for f in range(n_frames):
        pred_hnsw[query_idx] = I_t[f, 0]
        inds, dists = I_t[f], dist_for_hmm[f]
        if len(inds) < K:
            inds = np.concatenate([inds, np.full(K - len(inds), inds[0], dtype=np.int64)])
            dists = np.concatenate([dists, np.full(K - len(dists), dists[0], dtype=np.float64)])
        else:
            inds, dists = inds[:K], dists[:K]
        pred_hmm[query_idx] = hmm.update(inds, dists)
        query_idx += 1

assert query_idx == n_queries

In [6]:
correct_hnsw = np.array([pred_hnsw[i] in gt_9_list[i] for i in range(n_queries)])
correct_hmm = np.array([pred_hmm[i] in gt_9_list[i] for i in range(n_queries)])
valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)])
n_eval = int(np.sum(valid))

# 诊断：query 与预测分别属于哪个场景（全局检索时预测常落在其它场景导致 Recall=0）
query_to_scene = {}
for scene_name, start, end in trajectory_ranges:
    for i in range(start, end):
        query_to_scene[i] = scene_name
db_idx_to_scene = {}
for scene_name, start, end in db_scene_ranges:
    for j in range(start, end):
        db_idx_to_scene[j] = scene_name
valid_indices = np.where(valid)[0]
n_diag = min(5, len(valid_indices))
print("诊断（前 %d 条有 GT 的 query）：" % n_diag)
for k in range(n_diag):
    i = valid_indices[k]
    q_scene = query_to_scene.get(i, "?")
    p_hnsw = int(pred_hnsw[i])
    p_hmm = int(pred_hmm[i])
    p_hnsw_scene = db_idx_to_scene.get(p_hnsw, "?")
    p_hmm_scene = db_idx_to_scene.get(p_hmm, "?")
    gt_set = list(gt_9_list[i])[:5]
    in_hnsw = pred_hnsw[i] in gt_9_list[i]
    in_hmm = pred_hmm[i] in gt_9_list[i]
    print(f"  query {i}: 场景={q_scene} | HNSW pred={p_hnsw} (场景={p_hnsw_scene}) in_GT={in_hnsw} | HMM pred={p_hmm} (场景={p_hmm_scene}) in_GT={in_hmm} | GT 示例={gt_set}")
same_scene_hnsw = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hnsw[i]))
same_scene_hmm = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hmm[i]))
print(f"预测与 query 同场景的比例: HNSW {same_scene_hnsw}/{n_eval}, HMM {same_scene_hmm}/{n_eval}")
i0 = valid_indices[0]
p0 = int(pred_hnsw[i0])
gt0 = list(gt_9_list[i0])
print("")
print("query 0 细查（看预测与 GT 子图是否相邻）:")
print(f"  预测子图 db_names[{p0}] = {db_names[p0]}")
for g in sorted(gt0)[:5]:
    print(f"  GT 子图 db_names[{g}] = {db_names[g]}")
print("  （若 id_startx_starty 相差很大，说明坐标→瓦片或 query 与 uav_infos 行序可能不一致）")
print("前 3 条 query 的图像名（请与 uav_infos 前 3 行的 file_name 核对是否一一对应）:")
for k in range(min(3, len(query_names))):
    print(f"  query {k}: {query_names[k]}")
print("")

if n_eval > 0:
    recall_hnsw = correct_hnsw[valid].mean()
    recall_hmm = correct_hmm[valid].mean()
    print("Recall@25（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  仅 HNSW:      {recall_hnsw:.4f}  ({n_eval} 条 query)")
    print(f"  HNSW + HMM:   {recall_hmm:.4f}")
else:
    print("无有效坐标法 GT，请检查 uav_infos.csv 与 GDAL 路径。")

诊断（前 5 条有 GT 的 query）：
  query 0: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[129, 130, 131, 133, 134]
  query 1: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[129, 130, 131, 133, 134]
  query 2: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[129, 130, 131, 133, 134]
  query 3: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[129, 130, 131, 133, 134]
  query 4: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[129, 130, 131, 133, 134]
预测与 query 同场景的比例: HNSW 5988/5988, HMM 5988/5988

query 0 细查（看预测与 GT 子图是否相邻）:
  预测子图 db_names[175] = tif/259_1906_2206.tif
  GT 子图 db_names[107] = tif/198_1306_1756.tif
  GT 子图 db_names[108] = tif/199_1456_1756.tif
  GT 子图 db_names[111] = tif/200_1606_1756.tif
  GT 子图 db_names[112] = tif/201_1756_1756.tif
  GT 子图 db_names[113] = ti